# 04 — Vision with TorchVision: CNNs, Transfer Learning, Augmentations, Detection/Segmentation Patterns

Goal: build vision models and leverage pretrained backbones effectively.

_Generated: 2026-01-25_

## Setup

```bash
pip install torch torchvision torchaudio
pip install transformers datasets tokenizers accelerate evaluate
pip install matplotlib tensorboard
```

In [ ]:

import os, math, random
import numpy as np
import torch

def get_device():
    if torch.cuda.is_available():
        return torch.device("cuda")
    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")

device = get_device()
print("torch:", torch.__version__)
print("device:", device)

## 1. TorchVision availability and versions

In [ ]:

try:
    import torchvision
    import torchvision.transforms as T
    from torchvision import datasets, models
    print("torchvision:", torchvision.__version__)
except Exception as e:
    print("torchvision not available:", e)

## 2. Minimal CNN (28x28 grayscale)

In [ ]:

import torch, torch.nn as nn
import torch.nn.functional as F

class SmallCNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 16, 3, padding=1)
        self.conv2 = nn.Conv2d(16, 32, 3, padding=1)
        self.pool = nn.MaxPool2d(2)
        self.fc1 = nn.Linear(32*7*7, 128)
        self.fc2 = nn.Linear(128, num_classes)
    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = torch.flatten(x, 1)
        x = F.relu(self.fc1(x))
        return self.fc2(x)

m = SmallCNN().to(device)
m(torch.randn(8,1,28,28, device=device)).shape

## 3. Transfer learning template (ResNet)

Typical workflow:
- load pretrained backbone
- replace classifier head
- freeze backbone; train head
- unfreeze subset; fine-tune with smaller LR

In [ ]:

transfer_learning_template = r'''
import torch
import torch.nn as nn
from torchvision import models

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
num_classes = 5

model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
model.fc = nn.Linear(model.fc.in_features, num_classes)
model.to(device)

for name, p in model.named_parameters():
    if not name.startswith("fc."):
        p.requires_grad = False

optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=3e-4)
'''
print(transfer_learning_template)

## 4. Detection and segmentation patterns (overview)

Detection models require a target dict per image:
- boxes: FloatTensor[N,4]
- labels: Int64Tensor[N]
Segmentation uses masks.

You typically need a custom collate_fn returning lists of images/targets.